In [ ]:
from dataclasses import dataclass, field 
import json
from pathlib import Path



@dataclass
class DirNode:
    name: str
    children: dict = field(default_factory=dict)
    meta: dict = field(default_factory=dict)
    path: Path |None = None

print(DirNode(name="inputs"))

model= DirNode(name="Model")

inputs = DirNode(name="inputs", children={"Model": model})
print(inputs.children["Model"].name)

print(inputs)


DirNode(name='inputs', children={}, meta={}, path=None)
Model
DirNode(name='inputs', children={'Model': DirNode(name='Model', children={}, meta={}, path=None)}, meta={}, path=None)


In [23]:
####Directory JSON ####
json_path = Path("/home/tdeibert/Projects/Nuclear_Scaling/Repository_Structure.json")
with open(json_path) as f:
    structure = json.load(f) 

print(structure["data_root"]["Inputs"].keys())

META_KEYS= set(structure["_schema"]["metadata_keys"])
inputs_json = structure["data_root"]["Inputs"]

meta = {} 
children = {}

for key, value in inputs_json.items():
    if key in META_KEYS:
        meta[key] = value
    else:
        children[key] = value 

print("META", meta)
print("CHILDREN", list(children.keys()))

inputs_node = DirNode(name="Inputs", children=children, meta=meta)

print(inputs_node.meta["_desc"])

dict_keys(['_desc', '_machines', '_git', '_created_by', 'Models', 'Raw_Images', 'Training_Data'])
META {'_desc': 'Everything the pipeline and training read from', '_machines': ['local', 'cheaha'], '_git': 'n/a', '_created_by': 'scaffold'}
CHILDREN ['Models', 'Raw_Images', 'Training_Data']
Everything the pipeline and training read from


In [28]:
def build_node(name, node_json):
    meta = {}
    children = {}
    for key, value in node_json.items():
        if key in META_KEYS:
            meta[key] = value
        else:
            children[key] = build_node(key,value)
    return DirNode(name=name,children=children,meta=meta)

node = build_node("Inputs", structure["data_root"]["Inputs"])
code_tree = build_node("code_root", structure["code_root"])
data_tree = build_node("data_root", structure["data_root"])

print(node.meta["_desc"])
print(type(node.children["Models"]))
print(node.children["Raw_Images"].children["Condition"].meta["_desc"])
print(data_tree.children["Database"].children["Radial_Profiles"].meta["_machines"])
print(list(code_tree.children["Code_Repository"].children["Python"].children.keys()))
# One-off informational run: print one whole node to confirm the new path field exists
print(data_tree.children["Database"].children["Radial_Profiles"])

Everything the pipeline and training read from
<class '__main__.DirNode'>
One folder per experimental condition. Each holds the raw .tif files (named <experiment_id>, e.g. control_extract_1.1) and their Acquisition sidecar JSONs.
['local']
['Configs', 'Analysis', 'Image_Segmentation', 'Model_Training', 'Database', 'Utilities', 'Nuclear_Scaling_Core']
DirNode(name='Radial_Profiles', children={}, meta={'_desc': 'zstd parquet files holding bulk radial-profile rows; the database keeps only pointer rows to these', '_machines': ['local'], '_git': 'n/a', '_created_by': 'importer'}, path=None)
